# Basic Statistical Testing in Python

This notebook demonstrates how to perform basic hypothesis testing using the `scipy.stats` library. We will focus on the **Student's T-test**, which is used to determine if there is a significant difference between the means of two independent groups.

## Key Concepts:
- **Hypothesis Testing**: A statistical method used to make inferences about a population based on a sample.
- **Null Hypothesis (H0)**: The assumption that there is no effect or no difference between groups.
- **Alternative Hypothesis (H1)**: The assumption that there is a significant effect or difference.
- **P-value**: The probability of observing the results if the null hypothesis is true. A low p-value (typically < 0.05) suggests that we should reject the null hypothesis.


In [2]:
# We import numpy for numerical operations, pandas for data manipulation, 
# and the stats module from scipy for statistical tests.
import numpy as np
import pandas as pd
from scipy import stats

## Loading and Exploring the Dataset

We will use a dataset containing student grades for six assignments. Our goal is to see if students who submitted their first assignment early perform differently than those who submitted it later.


In [3]:
# Load the grades dataset from the datasets directory
df = pd.read_csv("datasets/grades.csv")

# Display the first few rows of the dataframe
df.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
0,B73F2C11-70F0-E37D-8B10-1D20AFED50B1,92.733946,2015-11-02 06:55:34.282000000,83.030552,2015-11-09 02:22:58.938000000,67.164441,2015-11-12 08:58:33.998000000,53.011553,2015-11-16 01:21:24.663000000,47.710398,2015-11-20 13:24:59.692000000,38.168318,2015-11-22 18:31:15.934000000
1,98A0FAE0-A19A-13D2-4BB5-CFBFD94031D1,86.790821,2015-11-29 14:57:44.429000000,86.290821,2015-12-06 17:41:18.449000000,69.772657,2015-12-10 08:54:55.904000000,55.098125,2015-12-13 17:32:30.941000000,49.588313,2015-12-19 23:26:39.285000000,44.629482,2015-12-21 17:07:24.275000000
2,D0F62040-CEB0-904C-F563-2F8620916C4E,85.512541,2016-01-09 05:36:02.389000000,85.512541,2016-01-09 06:39:44.416000000,68.410033,2016-01-15 20:22:45.882000000,54.728026,2016-01-11 12:41:50.749000000,49.255224,2016-01-11 17:31:12.489000000,44.329701,2016-01-17 16:24:42.765000000
3,FFDF2B2C-F514-EF7F-6538-A6A53518E9DC,86.030665,2016-04-30 06:50:39.801000000,68.824532,2016-04-30 17:20:38.727000000,61.942079,2016-05-12 07:47:16.326000000,49.553663,2016-05-07 16:09:20.485000000,49.553663,2016-05-24 12:51:18.016000000,44.598297,2016-05-26 08:09:12.058000000
4,5ECBEEB6-F1CE-80AE-3164-E45E99473FB4,64.813800,2015-12-13 17:06:10.750000000,51.491040,2015-12-14 12:25:12.056000000,41.932832,2015-12-29 14:25:22.594000000,36.929549,2015-12-28 01:29:55.901000000,33.236594,2015-12-29 14:46:06.628000000,33.236594,2016-01-05 01:06:59.546000000


In [5]:
print(f"There are {df.shape[0]} rows and {df.shape[1]} columns.")

There are 2315 rows and 13 columns.


## Splitting the Data: Early vs. Late Finishers

We define 'early finishers' as those who submitted their first assignment before the start of 2016.


In [6]:
# Create a boolean mask for students who submitted assignment 1 before 2016
earlyFinishers = df[pd.to_datetime(df["assignment1_submission"]) < "2016"]

# Preview the early finishers dataframe
earlyFinishers.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
0,B73F2C11-70F0-E37D-8B10-1D20AFED50B1,92.733946,2015-11-02 06:55:34.282000000,83.030552,2015-11-09 02:22:58.938000000,67.164441,2015-11-12 08:58:33.998000000,53.011553,2015-11-16 01:21:24.663000000,47.710398,2015-11-20 13:24:59.692000000,38.168318,2015-11-22 18:31:15.934000000
1,98A0FAE0-A19A-13D2-4BB5-CFBFD94031D1,86.790821,2015-11-29 14:57:44.429000000,86.290821,2015-12-06 17:41:18.449000000,69.772657,2015-12-10 08:54:55.904000000,55.098125,2015-12-13 17:32:30.941000000,49.588313,2015-12-19 23:26:39.285000000,44.629482,2015-12-21 17:07:24.275000000
4,5ECBEEB6-F1CE-80AE-3164-E45E99473FB4,64.813800,2015-12-13 17:06:10.750000000,51.491040,2015-12-14 12:25:12.056000000,41.932832,2015-12-29 14:25:22.594000000,36.929549,2015-12-28 01:29:55.901000000,33.236594,2015-12-29 14:46:06.628000000,33.236594,2016-01-05 01:06:59.546000000
5,D09000A0-827B-C0FF-3433-BF8FF286E15B,71.647278,2015-12-28 04:35:32.836000000,64.052550,2016-01-03 21:05:38.392000000,64.752550,2016-01-07 08:55:43.692000000,57.467295,2016-01-11 00:45:28.706000000,57.467295,2016-01-11 00:54:13.579000000,57.467295,2016-01-20 19:54:46.166000000
8,C9D51293-BD58-F113-4167-A7C0BAFCB6E5,66.595568,2015-12-25 02:29:28.415000000,52.916454,2015-12-31 01:42:30.046000000,48.344809,2016-01-05 23:34:02.180000000,47.444809,2016-01-02 07:48:42.517000000,37.955847,2016-01-03 21:27:04.266000000,37.955847,2016-01-19 15:24:31.060000000


In [8]:
# Late finishers are those not included in the earlyFinishers dataframe
lateFinishers = df[~df.index.isin(earlyFinishers.index)]

# Preview the late finishers dataframe
lateFinishers.head()

,student_id,assignment1_grade,assignment1_submission,assignment2_grade,assignment2_submission,assignment3_grade,assignment3_submission,assignment4_grade,assignment4_submission,assignment5_grade,assignment5_submission,assignment6_grade,assignment6_submission
2,D0F62040-CEB0-904C-F563-2F8620916C4E,85.512541,2016-01-09 05:36:02.389000000,85.512541,2016-01-09 06:39:44.416000000,68.410033,2016-01-15 20:22:45.882000000,54.728026,2016-01-11 12:41:50.749000000,49.255224,2016-01-11 17:31:12.489000000,44.329701,2016-01-17 16:24:42.765000000
3,FFDF2B2C-F514-EF7F-6538-A6A53518E9DC,86.030665,2016-04-30 06:50:39.801000000,68.824532,2016-04-30 17:20:38.727000000,61.942079,2016-05-12 07:47:16.326000000,49.553663,2016-05-07 16:09:20.485000000,49.553663,2016-05-24 12:51:18.016000000,44.598297,2016-05-26 08:09:12.058000000
6,3217BE3F-E4B0-C3B6-9F64-462456819CE4,87.498744,2016-03-05 11:05:25.408000000,69.998995,2016-03-09 07:29:52.405000000,55.999196,2016-03-16 22:31:24.316000000,50.399276,2016-03-18 07:19:26.032000000,45.359349,2016-03-19 10:35:41.869000000,45.359349,2016-03-23 14:02:00.987000000
7,F1CB5AA1-B3DE-5460-FAFF-BE951FD38B5F,80.576090,2016-01-24 18:24:25.619000000,72.518481,2016-01-27 13:37:12.943000000,65.266633,2016-01-30 14:34:36.581000000,65.266633,2016-02-03 22:08:49.002000000,65.266633,2016-02-16 14:22:23.664000000,65.266633,2016-02-18 08:35:04.796000000
9,E2C617C2-4654-622C-AB50-1550C4BE42A0,59.270882,2016-03-06 12:06:26.185000000,59.270882,2016-03-13 02:07:25.289000000,53.343794,2016-03-17 07:30:09.241000000,53.343794,2016-03-20 21:45:56.229000000,42.675035,2016-03-27 15:55:04.414000000,38.407532,2016-03-30 20:33:13.554000000


## Comparing Group Means

Let's look at the average grade for Assignment 1 for both groups.


In [9]:
print(earlyFinishers['assignment1_grade'].mean())
print(lateFinishers['assignment1_grade'].mean())

74.94728457024304
74.0450648477065


## Performing the T-Test

We use `ttest_ind` (independent t-test) to compare the means. The null hypothesis is that the groups have equal means.


In [12]:
from scipy.stats import ttest_ind

# Compare assignment 1 grades
print("Assignment 1:", ttest_ind(earlyFinishers['assignment1_grade'], lateFinishers['assignment1_grade']))

TtestResult(statistic=np.float64(1.3223540853721598), pvalue=np.float64(0.18618101101713846), df=np.float64(2313.0))


In [14]:
# Run the t-test for all other assignments to see if there is any broad trend
print("Assignment 2:", ttest_ind(earlyFinishers['assignment2_grade'], lateFinishers['assignment2_grade']))
print("Assignment 3:", ttest_ind(earlyFinishers['assignment3_grade'], lateFinishers['assignment3_grade']))
print("Assignment 4:", ttest_ind(earlyFinishers['assignment4_grade'], lateFinishers['assignment4_grade']))
print("Assignment 5:", ttest_ind(earlyFinishers['assignment5_grade'], lateFinishers['assignment5_grade']))
print("Assignment 6:", ttest_ind(earlyFinishers['assignment6_grade'], lateFinishers['assignment6_grade']))

TtestResult(statistic=np.float64(1.2514717608216366), pvalue=np.float64(0.21088896270044238), df=np.float64(2313.0))
TtestResult(statistic=np.float64(1.6133726558705392), pvalue=np.float64(0.1067999810222786), df=np.float64(2313.0))
TtestResult(statistic=np.float64(0.049671157386456125), pvalue=np.float64(0.9603887297893369), df=np.float64(2313.0))
TtestResult(statistic=np.float64(-0.05279315545404755), pvalue=np.float64(0.9579012739746491), df=np.float64(2313.0))
TtestResult(statistic=np.float64(-0.11609743352612056), pvalue=np.float64(0.9075854011989657), df=np.float64(2313.0))


## Simulating the Dangers of P-Hacking

If we run many tests, we are likely to find a 'statistically significant' result purely by chance. This is known as p-hacking or the multiple comparisons problem.

In the examples below, we generate 100 columns of random data and compare two dataframes. Even though there is NO real difference, we expect about 10% of results to be 'significant' at alpha=0.1.


In [25]:
df1 = pd.DataFrame([np.random.random(100) for x in range(100)])

df1.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.612959,0.576916,0.722862,0.021049,0.799689,0.847767,0.467600,0.897130,0.446103,0.817312,...,0.287787,0.389621,0.571446,0.198539,0.424570,0.074667,0.938565,0.429493,0.818609,0.559168
1,0.058822,0.435364,0.671377,0.016056,0.190963,0.888377,0.387068,0.909093,0.594111,0.859313,...,0.963742,0.959739,0.909497,0.667005,0.098857,0.196079,0.145914,0.945893,0.547212,0.359846
2,0.427809,0.140626,0.262786,0.242636,0.352811,0.842171,0.891931,0.930220,0.857809,0.673680,...,0.308721,0.553727,0.563616,0.929150,0.539082,0.940840,0.196110,0.852195,0.175970,0.184997
3,0.202011,0.903300,0.926903,0.023643,0.230372,0.718506,0.029587,0.904113,0.844620,0.977381,...,0.044343,0.877701,0.981909,0.472032,0.975845,0.672906,0.183218,0.934628,0.274003,0.882905
4,0.892221,0.076200,0.694615,0.739562,0.900684,0.527618,0.350162,0.357962,0.912035,0.548882,...,0.979529,0.123177,0.167969,0.403211,0.353276,0.991709,0.779597,0.355519,0.162210,0.293913


In [26]:
df2 = pd.DataFrame([np.random.random(100) for x in range(100)])

df2.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.681279,0.950242,0.939610,0.935595,0.985430,0.379119,0.160032,0.612699,0.141170,0.260333,...,0.857338,0.572512,0.303550,0.876813,0.003373,0.187672,0.141937,0.419212,0.060377,0.969436
1,0.598195,0.748984,0.207770,0.220614,0.080395,0.338021,0.305039,0.714371,0.888050,0.181893,...,0.934528,0.820054,0.944773,0.802541,0.721857,0.472452,0.487362,0.589881,0.262380,0.274135
2,0.126956,0.390733,0.149223,0.315135,0.951062,0.239937,0.340352,0.191871,0.847430,0.630532,...,0.939362,0.546676,0.228605,0.102052,0.815096,0.505236,0.060685,0.966500,0.369048,0.280283
3,0.520133,0.239994,0.902526,0.459388,0.219518,0.864525,0.514718,0.458303,0.541772,0.260643,...,0.631311,0.705552,0.051079,0.006790,0.769220,0.824876,0.821471,0.178360,0.087962,0.066991
4,0.477501,0.864678,0.826286,0.037970,0.714890,0.812778,0.549901,0.570319,0.460811,0.138270,...,0.230418,0.968160,0.063615,0.726885,0.406466,0.430985,0.579102,0.067838,0.260673,0.282096


In [27]:
def test_columns(alpha=0.1):
    num_diff=0
    for col in df1.columns:
        teststat,pval=ttest_ind(df1[col],df2[col])
        if pval<=alpha:
            print("Col {} is statistically significantly different at alpha={}, pval={}".format(col,alpha,pval))
            num_diff=num_diff+1
    print("Total number different was {}, which is {}%".format(num_diff,float(num_diff)/len(df1.columns)*100))

test_columns()

Col 2 is statistically significantly different at alpha=0.1, pval=0.021914439224866692
Col 24 is statistically significantly different at alpha=0.1, pval=0.0001433798066675946
Col 29 is statistically significantly different at alpha=0.1, pval=0.03485685552237431
Col 36 is statistically significantly different at alpha=0.1, pval=0.07835564932647875
Col 49 is statistically significantly different at alpha=0.1, pval=0.0888687197350077
Col 50 is statistically significantly different at alpha=0.1, pval=0.09722830513804115
Col 59 is statistically significantly different at alpha=0.1, pval=0.05721925900009301
Col 62 is statistically significantly different at alpha=0.1, pval=0.09854463339809827
Col 65 is statistically significantly different at alpha=0.1, pval=0.09016114174085085
Col 67 is statistically significantly different at alpha=0.1, pval=0.04573536804702109
Col 70 is statistically significantly different at alpha=0.1, pval=0.039381785619303215
Col 76 is statistically significantly dif

In [28]:
test_columns(0.05)

Col 2 is statistically significantly different at alpha=0.05, pval=0.021914439224866692
Col 24 is statistically significantly different at alpha=0.05, pval=0.0001433798066675946
Col 29 is statistically significantly different at alpha=0.05, pval=0.03485685552237431
Col 67 is statistically significantly different at alpha=0.05, pval=0.04573536804702109
Col 70 is statistically significantly different at alpha=0.05, pval=0.039381785619303215
Col 76 is statistically significantly different at alpha=0.05, pval=0.028749829832786605
Total number different was 6, which is 6.0%


## Testing Different Distributions

If the underlying distributions are actually different (e.g., normal vs. chi-square), the T-test will correctly identify significance in almost every case.


In [29]:
df2=pd.DataFrame([np.random.chisquare(df=1,size=100) for x in range(100)])
test_columns()

Col 0 is statistically significantly different at alpha=0.1, pval=0.00341165072738544
Col 1 is statistically significantly different at alpha=0.1, pval=3.568639916885418e-05
Col 2 is statistically significantly different at alpha=0.1, pval=0.04817532696513629
Col 3 is statistically significantly different at alpha=0.1, pval=0.0002134825139120945
Col 4 is statistically significantly different at alpha=0.1, pval=0.0010606110108538388
Col 5 is statistically significantly different at alpha=0.1, pval=0.0056317024979216776
Col 6 is statistically significantly different at alpha=0.1, pval=0.0027358290463920464
Col 7 is statistically significantly different at alpha=0.1, pval=4.28778021225563e-05
Col 8 is statistically significantly different at alpha=0.1, pval=5.104374094719542e-08
Col 9 is statistically significantly different at alpha=0.1, pval=0.013511304634726851
Col 10 is statistically significantly different at alpha=0.1, pval=0.00010232669580270631
Col 11 is statistically significantl